In [1]:
# 02_credit_core.ipynb
#
# Experiment 2 -- Two axes of bias in real credit data: outcome parity vs proxy leakage.
#
# Purpose:
#   On three real credit datasets (German, Taiwan, GiveMeSomeCredit), measure two
#   DISTINCT things that are usually conflated:
#     (1) outcome parity  -- Disparate Impact Ratio (directional) + equalized-odds gaps;
#     (2) proxy leakage    -- how well the protected attribute is reconstructed from
#                             the remaining features (reconstruction ROC-AUC).
#   A model can satisfy parity yet still carry the protected attribute through proxies.
#   Conditions: with_protected (attribute IS a feature), drop_protected (attribute
#   removed, proxies kept), drop_proxies (attribute + top proxies removed),
#   gov_engine (group-wise threshold calibration). 30 seeds, 95% CIs.
#   Protected attributes: German=age(<25); Taiwan=sex,age,marriage; GMSC=age.
#
# Outputs (PNG+PDF, 600 dpi, grayscale, no captions; paths not printed):
#   results/tables/exp2_credit_core.csv
#   results/figures/exp2_<dataset>_<attr>_parity_leakage.(png|pdf)

import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
FIG_DIR  = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR  = os.path.join(PROJECT_ROOT, "results", "tables")
for _d in (DATA_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(_d, exist_ok=True)

sns.set_theme(style="whitegrid")
GRAYS = ["#000000", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYS))
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "grid.color": "0.85"})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

def dir_ratio(y_pred, group):
    approve = (y_pred == 0).astype(int)                 # favorable = non-default
    r1 = approve[group==1].mean(); r0 = approve[group==0].mean()
    return (r1/r0) if r0 > 0 else np.nan                # directional four-fifths

def eo_gaps(y_true, y_pred, group):
    def rate(ct, mask):
        idx=(y_true==ct)&mask
        return (y_pred[idx]==1).mean() if idx.sum() else np.nan
    a,b = group==1, group==0
    return abs(rate(1,a)-rate(1,b)), abs(rate(0,a)-rate(0,b))

def leakage_auc(Xdf, g, seed):
    # Reconstruct the protected group from a feature set; higher AUC = more leakage.
    Xf = Xdf.fillna(Xdf.median(numeric_only=True)).values
    Xs = StandardScaler().fit_transform(Xf)
    try:
        return cross_val_score(LogisticRegression(max_iter=1000), Xs, g, cv=3,
                               scoring="roc_auc").mean()
    except Exception:
        return np.nan

# ---- Loaders: return (features_df, protected_series, y_default[1=default], group[1=disadvantaged]) ----
def load_german(attr):
    df = pd.read_csv(os.path.join(DATA_DIR, "german_credit.csv"))
    y = (df["credit_risk"] == 0).astype(int).values
    g = (df["age"] < 25).astype(int).values
    cat = [c for c in df.select_dtypes(include=["object","string"]).columns]
    base = pd.get_dummies(df.drop(columns=["credit_risk","age"]), columns=cat).astype(float)
    prot = pd.Series(g, name="PROTECTED", index=base.index).astype(float)
    return base, prot, y, g

def load_taiwan(attr):
    df = pd.read_parquet(os.path.join(DATA_DIR, "taiwan_credit.parquet"))
    y = df["DEFAULT"].astype(int).values
    if attr == "sex":        g = (df["SEX"] == 2).astype(int).values
    elif attr == "age":      g = (df["AGE"] < 30).astype(int).values
    else:                    g = (df["MARRIAGE"] == 2).astype(int).values  # single vs not
    drop = ["DEFAULT","VIP","SEX_STD","AGE_YEARS","SEX","AGE","MARRIAGE"]
    base = df.drop(columns=[c for c in drop if c in df.columns]).astype(float).reset_index(drop=True)
    prot = pd.Series(g, name="PROTECTED").astype(float)
    return base, prot, y, g

def load_gmsc(attr):
    df = pd.read_parquet(os.path.join(DATA_DIR, "gmsc_credit.parquet"))
    y = df["DEFAULT"].astype(int).values
    g = (df["AGE_YEARS"] < 40).astype(int).values
    drop = ["DEFAULT","VIP","AGE_YEARS"]
    base = df.drop(columns=[c for c in drop if c in df.columns]).astype(float).reset_index(drop=True)
    base = base.fillna(base.median(numeric_only=True))
    prot = pd.Series(g, name="PROTECTED").astype(float)
    return base, prot, y, g

def top_proxies(base, g, k=4):
    c = base.apply(lambda col: abs(np.corrcoef(col, g)[0,1]) if col.std()>0 else 0.0)
    return c.sort_values(ascending=False).head(k).index.tolist()

def one_run(base, prot, y, g, cond, seed, subsample=None):
    idx = np.arange(len(y))
    if subsample and len(y) > subsample:
        idx = np.random.default_rng(seed).choice(idx, subsample, replace=False)
    b = base.iloc[idx].reset_index(drop=True)
    pr = prot.iloc[idx].reset_index(drop=True)
    ys, gs = y[idx], g[idx]
    proxies = top_proxies(b, gs, k=4)
    # assemble the condition's feature frame
    if cond == "with_protected":
        X = pd.concat([b, pr], axis=1); feats = X.columns.tolist()
    elif cond in ("drop_protected", "gov_engine"):
        X = b; feats = b.columns.tolist()
    else:  # drop_proxies
        X = b; feats = [c for c in b.columns if c not in proxies]
    Xtr, Xte, ytr, yte, gtr, gte = train_test_split(
        X[feats], ys, gs, test_size=0.3, random_state=seed, stratify=ys)
    sc = StandardScaler().fit(Xtr)
    m = LogisticRegression(max_iter=2000).fit(sc.transform(Xtr), ytr)
    p = m.predict_proba(sc.transform(Xte))[:,1]
    auc = roc_auc_score(yte, p)
    if cond != "gov_engine":
        yp = (p >= 0.5).astype(int)
    else:
        base_rate = ((p >= 0.5).astype(int) == 0).mean()
        t1 = np.quantile(p[gte==1], base_rate); t0 = np.quantile(p[gte==0], base_rate)
        yp = np.where(gte==1, (p>=t1), (p>=t0)).astype(int)
    dt, dfp = eo_gaps(yte, yp, gte)
    # Proxy leakage: reconstruct the protected group from THIS condition's feature
    # set, over the whole subsample (independent of the train/test split).
    leak = leakage_auc(X[feats], gs, seed) if cond in ("drop_protected","drop_proxies") else np.nan
    return auc, dir_ratio(yp, gte), dt, dfp, leak

DATASETS = [("german","age",load_german,None),
            ("taiwan","sex",load_taiwan,None),
            ("taiwan","age",load_taiwan,None),
            ("taiwan","marriage",load_taiwan,None),
            ("gmsc","age",load_gmsc,20000)]
CONDS = ["with_protected","drop_protected","drop_proxies","gov_engine"]
SEEDS = list(range(30))

rows = []
for dsname, attr, loader, subs in DATASETS:
    base, prot, y, g = loader(attr)
    # base disparity in the data itself (model-free)
    appr1 = (1-y[g==1]).mean(); appr0 = (1-y[g==0]).mean()
    base_data_dir = appr1/appr0 if appr0>0 else np.nan
    for cond in CONDS:
        arr = np.array([one_run(base, prot, y, g, cond, s, subs) for s in SEEDS])
        m = np.nanmean(arr,0); ci = 1.96*np.nanstd(arr,0)/np.sqrt(len(SEEDS))
        rows.append({"dataset":dsname,"protected":attr,"condition":cond,
                     "base_data_DIR":round(base_data_dir,3),
                     "AUC":m[0],"DIR":m[1],"DIR_ci":ci[1],
                     "dTPR":m[2],"dFPR":m[3],"leak_AUC":m[4]})

res = pd.DataFrame(rows)
res.to_csv(os.path.join(TAB_DIR, "exp2_credit_core.csv"), index=False)
print(res.drop(columns=["DIR_ci"]).round(3).to_string(index=False))

# Figure per dataset/attr: DIR bars (left axis) with leakage noted in drop_protected
order = CONDS
for (dsname, attr) in res[["dataset","protected"]].drop_duplicates().itertuples(index=False):
    sub = res[(res.dataset==dsname)&(res.protected==attr)].set_index("condition").loc[order].reset_index()
    fig, ax = plt.subplots(figsize=(6,4.2))
    ax.bar(range(len(sub)), sub["DIR"], yerr=sub["DIR_ci"], capsize=4,
           edgecolor="black", linewidth=1.0, color=GRAYS)
    ax.axhline(0.8, color="black", linestyle="--", linewidth=0.9)
    ax.set_xticks(range(len(sub))); ax.set_xticklabels(sub["condition"], rotation=20, ha="right")
    ax.set_ylabel("Disparate Impact Ratio (DIR)"); ax.set_ylim(0, 1.2)
    fig.tight_layout(); save_fig(fig, f"exp2_{dsname}_{attr}_parity_leakage"); plt.close(fig)

print("Done: credit-core two-axis experiment, 3 datasets, 30 seeds.")


dataset protected      condition  base_data_DIR   AUC   DIR  dTPR  dFPR  leak_AUC
 german       age with_protected          0.821 0.778 0.806 0.115 0.165       NaN
 german       age drop_protected          0.821 0.778 0.835 0.102 0.137     0.800
 german       age   drop_proxies          0.821 0.775 0.919 0.119 0.101     0.765
 german       age     gov_engine          0.821 0.778 0.992 0.170 0.052       NaN
 taiwan       sex with_protected          1.045 0.720 1.033 0.048 0.015       NaN
 taiwan       sex drop_protected          1.045 0.720 1.014 0.016 0.008     0.546
 taiwan       sex   drop_proxies          1.045 0.668 1.007 0.018 0.003     0.538
 taiwan       sex     gov_engine          1.045 0.720 1.000 0.039 0.003       NaN
 taiwan       age with_protected          0.986 0.720 0.999 0.020 0.003       NaN
 taiwan       age drop_protected          0.986 0.720 0.977 0.054 0.010     0.677
 taiwan       age   drop_proxies          0.986 0.718 0.981 0.043 0.007     0.593
 taiwan       ag

Done: credit-core two-axis experiment, 3 datasets, 30 seeds.
